# DS 4320 Project 1 Pipeline: Do Demographics Improve Movie Recommendation Systems?

This notebook runs the full project pipeline in one place:
1. prepare the database in DuckDB
2. query and engineer analysis features
3. fit baseline and enhanced regression models
4. visualize the final comparison

## 1. Imports and Setup

The notebook uses the refactored pipeline scripts so the same logic can be reused in both batch files and notebook cells.

In [1]:
from pathlib import Path
import importlib.util

BASE = Path(".")

def load_module(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

prep = load_module("prep_module", str(BASE / "01_prep_data_refactored.py"))
analysis = load_module("analysis_module", str(BASE / "02_analysis_refactored.py"))
viz = load_module("viz_module", str(BASE / "03_visualization_refactored.py"))

## 2. Data Preparation

This step downloads MovieLens, creates synthetic demographics, injects a small controlled demographic signal into ratings, and loads the final tables into DuckDB.

In [2]:
prep_output = prep.run_prep(cleanup=False, export_parquet_files=True)
prep_output

{'db_path': 'data/movielens.db',
 'table_counts': {'ratings': 32000204,
  'movies': 87585,
  'tags': 2000072,
  'links': 87585,
  'users_demo': 200948},
 'data_dir': 'data'}

## 3. Validate the DuckDB database

These quick checks show that the relational database was created successfully.

In [3]:
import duckdb

con = duckdb.connect(prep_output["db_path"])
con.execute("SHOW TABLES").fetchdf()

,name
0,links
1,movies
2,ratings
3,tags
4,users_demo


In [4]:
for table in ["ratings", "movies", "tags", "links", "users_demo"]:
    print(table)
    display(con.execute(f"SELECT * FROM {table} LIMIT 5").fetchdf())

ratings


,userId,movieId,rating,timestamp
0,1,17,4.300013,944249077
1,1,25,1.421444,944250228
2,1,29,2.367940,943230976
3,1,30,5.000000,944249077
4,1,32,4.904073,943228858


movies


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


tags


,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


links


,movieId,imdbId,tmdbId
0,1,0114709,862
1,2,0113497,8844
2,3,0113228,15602
3,4,0114885,31357
4,5,0113041,11862


users_demo


,userId,age_group,sex,synth_source
0,1,35-44,Female,ACS_sampled
1,2,25-34,Male,ACS_sampled
2,3,25-34,Female,ACS_sampled
3,4,25-34,Female,ACS_sampled
4,5,35-44,Male,ACS_sampled


## 4. Query Preparation

This query joins the behavioral tables and the synthetic demographic table to create the analysis dataset.

In [5]:
query = '''
SELECT
    r.userId,
    r.movieId,
    r.rating,
    r.timestamp,
    m.genres,
    u.age_group,
    u.sex
FROM ratings r
JOIN movies m ON r.movieId = m.movieId
JOIN users_demo u ON r.userId = u.userId
'''
print(query)
joined_preview = con.execute(query + " LIMIT 5").fetchdf()
joined_preview


SELECT
    r.userId,
    r.movieId,
    r.rating,
    r.timestamp,
    m.genres,
    u.age_group,
    u.sex
FROM ratings r
JOIN movies m ON r.movieId = m.movieId
JOIN users_demo u ON r.userId = u.userId



,userId,movieId,rating,timestamp,genres,age_group,sex
0,1,17,4.300013,944249077,Drama|Romance,35-44,Female
1,1,25,1.421444,944250228,Drama|Romance,35-44,Female
2,1,29,2.367940,943230976,Adventure|Drama|Fantasy|Mystery|Sci-Fi,35-44,Female
3,1,30,5.000000,944249077,Crime|Drama,35-44,Female
4,1,32,4.904073,943228858,Mystery|Sci-Fi|Thriller,35-44,Female


## 5. Analysis Rationale

The baseline model uses only behavioral data, while the enhanced model adds age group and sex. Ridge regression is used because the project predicts a continuous rating and includes many correlated features after genre expansion. Continuous numeric features are standardized, binary genre columns are passed through directly, and categorical demographics are one-hot encoded.

## 6. Solution Analysis

Run the modeling pipeline and compare the baseline and enhanced models using RMSE, MAE, and R².

In [6]:
analysis_output = analysis.run_analysis()
analysis_output["results"]

,model,RMSE,MAE,R2
0,baseline,0.846442,0.646342,0.345447
1,enhanced,0.846440,0.646341,0.345450


In [7]:
analysis_output["baseline_coefficients"].head(10)

,feature,coefficient
2,user_mean_rating,0.416810
0,movie_mean_rating,0.396599
11,genre_Documentary,0.052983
16,genre_IMAX,-0.051963
3,user_rating_count,0.029981
15,genre_Horror,0.021353
1,movie_rating_count,-0.020358
12,genre_Drama,0.017648
17,genre_Musical,0.013010
20,genre_Sci-Fi,-0.009924


In [8]:
analysis_output["enhanced_coefficients"].head(15)

,feature,coefficient
2,user_mean_rating,0.416726
0,movie_mean_rating,0.396611
11,genre_Documentary,0.052965
16,genre_IMAX,-0.051946
3,user_rating_count,0.030021
15,genre_Horror,0.021352
1,movie_rating_count,-0.020350
12,genre_Drama,0.017644
17,genre_Musical,0.013008
20,genre_Sci-Fi,-0.009917


## 7. Visualization

The final figure focuses on RMSE because the project question is whether demographics provide a meaningful improvement in predictive accuracy. A tight y-axis range makes the negligible difference between the two models visible.

In [ ]:
import plotly.io as pio
pio.renderers.default = "browser"
fig = viz.run_visualization()
fig

## 8. Interpretation

The enhanced model produces virtually identical performance to the baseline model. Behavioral features such as user and movie average ratings dominate the coefficients, while demographic features contribute little to no measurable improvement in recommendation quality.